# Lesson 2.5 — End User Monitoring (EUM)

## Learning Objectives
- Understand what End User Monitoring captures (Browser RUM, Mobile APM)
- Describe user experience categories in AppDynamics EUM
- Analyze page performance and geographic distribution
- Understand how EUM data correlates to server-side BT data

**Reference**: [End User Monitoring](https://help.splunk.com/en/appdynamics-saas/end-user-monitoring/26.2.0/end-user-monitoring/overview-of-end-user-monitoring)

## 1. What is End User Monitoring?

Server-side APM tells you what happens *inside* your servers. But what about the user's actual experience?

**End User Monitoring (EUM)** captures:
- How long the page actually loads in the user's browser
- Network latency from the user's location
- JavaScript errors in the browser
- Mobile app crashes and performance
- User session data (pages visited, clicks, navigation paths)

AppDynamics EUM has two sub-products:

| Sub-product | Instrumentation | Data Collected |
|-------------|-----------------|----------------|
| **Browser RUM** | JavaScript snippet injected in HTML | Page load time, AJAX calls, JS errors, geographic data |
| **Mobile APM** | iOS/Android SDK | App load, screen transitions, network requests, crashes |

## 2. Browser RUM — How It Works

A tiny JavaScript agent (`adrum.js`) is inserted into your web pages. It runs in the user's browser and:

1. Measures the **Navigation Timing** (using the browser's Performance API):
   - DNS lookup time
   - TCP connection time
   - SSL handshake time
   - Time to First Byte (TTFB)
   - DOM processing time
   - DOM content loaded
   - Page fully loaded

2. Tracks **AJAX calls** made by the page:
   - URL, response code, timing
   - Correlation to server-side BTs

3. Captures **JavaScript errors**:
   - Error message, stack trace, URL, line number

4. Records **user actions**:
   - Virtual page changes (single-page apps)
   - Session duration

5. Collects **geographic and device data**:
   - Country, city (from IP)
   - Browser type, OS, device type

In [ ]:
# Simulate Browser RUM page performance data

import random, statistics
from collections import defaultdict

random.seed(13)

def simulate_page_loads(n: int, base_load_ms: float, location: str) -> list:
    """Simulate n browser page load records."""
    records = []
    for _ in range(n):
        dns    = random.gauss(20, 5)
        tcp    = random.gauss(30, 8)
        ssl    = random.gauss(45, 10)
        ttfb   = random.gauss(base_load_ms * 0.3, 50)
        dom    = random.gauss(base_load_ms * 0.5, 80)
        total  = dns + tcp + ssl + ttfb + dom
        records.append({
            "location": location,
            "dns_ms": max(1, dns),
            "tcp_ms": max(1, tcp),
            "ssl_ms": max(1, ssl),
            "ttfb_ms": max(50, ttfb),
            "dom_ms": max(100, dom),
            "total_ms": max(200, total),
            "js_error": random.random() < 0.03  # 3% JS error rate
        })
    return records

# Simulate users from different geographic locations
all_records = (
    simulate_page_loads(400, 1200, "US-East") +
    simulate_page_loads(180, 1800, "EU-West") +   # higher latency from Europe
    simulate_page_loads(120, 2500, "APAC")        # highest latency from Asia-Pacific
)

# Group by location and compute statistics
by_location = defaultdict(list)
for r in all_records:
    by_location[r["location"]].append(r)

print("Browser RUM — Page Load Performance by Geography")
print("=" * 80)
print(f"{'Location':<12} {'Users':>6} {'Avg Load':>10} {'P75':>8} {'P95':>8} {'JS Error%':>10}")
print("-" * 80)

for loc, records in sorted(by_location.items()):
    totals = sorted(r['total_ms'] for r in records)
    avg    = statistics.mean(totals)
    p75    = totals[int(len(totals) * 0.75)]
    p95    = totals[int(len(totals) * 0.95)]
    err_pct = sum(1 for r in records if r['js_error']) / len(records) * 100
    print(f"{loc:<12} {len(records):>6} {avg:>10.0f}ms {p75:>8.0f}ms {p95:>8.0f}ms {err_pct:>10.1f}%")

## 3. User Experience Classification

AppDynamics EUM classifies user experiences using the **Apdex** (Application Performance Index) model:

| Experience | Condition | Apdex Score |
|------------|-----------|-------------|
| **Satisfactory** | Load time ≤ threshold T | 1.0 |
| **Tolerating** | T < Load time ≤ 4T | 0.5 |
| **Frustrated** | Load time > 4T | 0.0 |

**Apdex formula**: `Score = (Satisfactory + Tolerating/2) / Total`

A score of 1.0 = perfect; below 0.7 = poor experience.

AppDynamics also has its own user experience labels:
- `NORMAL` — within acceptable thresholds
- `SLOW` — between slow and very slow threshold
- `VERY_SLOW` — above very slow threshold
- `STALL` — no response
- `ERROR` — JS or network error

In [ ]:
def compute_apdex(load_times: list, threshold_ms: float) -> float:
    """Compute the Apdex score for a list of page load times."""
    satisfactory = sum(1 for t in load_times if t <= threshold_ms)
    tolerating   = sum(1 for t in load_times if threshold_ms < t <= threshold_ms * 4)
    total = len(load_times)
    if total == 0:
        return 1.0
    return (satisfactory + tolerating / 2) / total

THRESHOLD_MS = 2000  # 2 second Apdex threshold

print(f"Apdex Report (threshold = {THRESHOLD_MS}ms)")
print("-" * 40)
for loc, records in sorted(by_location.items()):
    times  = [r['total_ms'] for r in records]
    score  = compute_apdex(times, THRESHOLD_MS)
    rating = "Excellent" if score >= 0.94 else ("Good" if score >= 0.85 else ("Fair" if score >= 0.7 else "Poor"))
    bar    = "█" * int(score * 20)
    print(f"{loc:<12} Apdex={score:.3f}  [{rating}]  {bar}")

print()
print("Action needed: APAC users have poor experience — consider CDN edge nodes or regional deployment.")

## 4. EUM ↔ Server-Side Correlation

AppDynamics links browser sessions to server-side Business Transactions:

- When `adrum.js` makes an AJAX call to `/api/checkout`, it injects a correlation header
- The App Server Agent reads the header and includes EUM context in the BT snapshot
- In the Controller UI, you can click from a **slow page load** directly to the **server-side transaction snapshot**

This end-to-end correlation is one of AppDynamics' most powerful features for root cause analysis.

### Mobile APM (brief overview)

For native iOS/Android apps, embed the **Mobile SDK** (iOS: CocoaPods/SPM, Android: Gradle).

Mobile APM tracks:
- App startup time
- Screen rendering performance
- Network requests (URL, response time, status code)
- Crash reports with full stack trace
- Breadcrumbs (custom events leading up to a crash)

All mobile data is stored in the **Events Service** and queryable via ADQL (covered in Phase 3).

In [ ]:
# EXERCISE 2.5: Analyze EUM data to find the worst-performing pages
#
# TASK: Given a list of page load records (each with a 'page' field),
# compute the Apdex score and average load time per page.
# Return a sorted list from worst to best Apdex.

import random
random.seed(2025)

APDEX_THRESHOLD_MS = 1500

pages = ["/home", "/checkout", "/search", "/product/{id}", "/account/orders"]
page_loads = []

base_times = {"/home": 800, "/checkout": 1800, "/search": 600,
              "/product/{id}": 1200, "/account/orders": 2400}

for page, base in base_times.items():
    for _ in range(random.randint(50, 200)):
        page_loads.append({
            "page": page,
            "load_ms": max(100, random.gauss(base, base * 0.3))
        })

def analyze_pages(records: list, threshold_ms: float) -> list:
    """
    TASK: Group records by page, compute Apdex and avg load time.
    Return list of dicts sorted by Apdex ascending (worst first):
    [{'page': str, 'count': int, 'avg_load_ms': float, 'apdex': float}, ...]
    """
    # TODO: Implement
    pass

results = analyze_pages(page_loads, APDEX_THRESHOLD_MS)

if results:
    print(f"Page Performance Analysis (threshold={APDEX_THRESHOLD_MS}ms):")
    print(f"{'Page':<22} {'Requests':>9} {'Avg Load':>9} {'Apdex':>7}")
    print("-" * 52)
    for r in results:
        print(f"{r['page']:<22} {r['count']:>9} {r['avg_load_ms']:>9.0f}ms {r['apdex']:>7.3f}")
else:
    print("Implement analyze_pages() to see the results.")

## Summary

- **Browser RUM** (`adrum.js`) captures client-side page performance: DNS, TTFB, DOM load, JS errors
- **Mobile APM** (iOS/Android SDK) captures app performance, network requests, and crash reports
- **Apdex** scores quantify user experience: Satisfactory (≤T), Tolerating (≤4T), Frustrated (>4T)
- EUM data correlates to server-side BTs — you can trace from a slow page load to a server call graph
- EUM data is stored in the **Events Service** and queryable via ADQL (covered in Phase 3)

---

**Phase 2 Complete!** Proceed to [Phase 3 — Analytics & ADQL](../phase-3-analytics-adql/lesson-3.1-business-iq-analytics-overview.ipynb)